In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.sparse as sp
import pickle

In [2]:
INPUT_DIR = Path('/kaggle/input/datasets/artembez/instacart-data')
OUTPUT_DIR = Path('/kaggle/working/')

departments = pd.read_csv(INPUT_DIR / 'departments.csv')
aisles = pd.read_csv(INPUT_DIR / 'aisles.csv')
orders = pd.read_csv(INPUT_DIR / 'orders.csv')
products = pd.read_csv(INPUT_DIR / 'products.csv')
order_products_prior = pd.read_csv(INPUT_DIR / 'order_products__prior.csv')
order_products_train = pd.read_csv(INPUT_DIR / 'order_products__train.csv')

### Сделаем матрицу взаимодействий

In [3]:
orders

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0
...,...,...,...,...,...,...,...
3421078,2266710,206209,prior,10,5,18,29.0
3421079,1854736,206209,prior,11,4,10,30.0
3421080,626363,206209,prior,12,1,12,18.0
3421081,2977660,206209,prior,13,1,12,7.0


In [4]:
order_products_train

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
...,...,...,...,...
1384612,3421063,14233,3,1
1384613,3421063,35548,4,1
1384614,3421070,35951,1,1
1384615,3421070,16953,2,1


In [5]:
# Объединим таблицы заказов
order_data = orders.merge(order_products_train, on = 'order_id')
order_data

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,1187899,1,train,11,4,8,14.0,196,1,1
1,1187899,1,train,11,4,8,14.0,25133,2,1
2,1187899,1,train,11,4,8,14.0,38928,3,1
3,1187899,1,train,11,4,8,14.0,26405,4,1
4,1187899,1,train,11,4,8,14.0,39657,5,1
...,...,...,...,...,...,...,...,...,...,...
1384612,272231,206209,train,14,6,14,30.0,40603,4,0
1384613,272231,206209,train,14,6,14,30.0,15655,5,0
1384614,272231,206209,train,14,6,14,30.0,42606,6,0
1384615,272231,206209,train,14,6,14,30.0,37966,7,0


In [6]:
# Посчитаем число купленных продуктов для каждого пользователя
user_products_count = order_data.groupby('user_id')['product_id'].count()
user_products_count

user_id
1         11
2         31
5          9
7          9
8         18
          ..
206199    22
206200    19
206203    13
206205    19
206209     8
Name: product_id, Length: 131209, dtype: int64

In [7]:
# оставим тех, у кого от 5 до 50 заказов (валидные для коллаборативной фильтрации)
min_products = 5
max_products = 50
valid_users = user_products_count[
(user_products_count >= min_products) &
(user_products_count <= max_products)
].index
valid_users

Index([     1,      2,      5,      7,      8,      9,     13,     14,     17,
           18,
       ...
       206191, 206193, 206195, 206196, 206198, 206199, 206200, 206203, 206205,
       206209],
      dtype='int64', name='user_id', length=100583)

In [8]:
# выберем из них 30 тысяч пользователей
selected_users = np.random.choice(valid_users, 30000, replace = False)
selected_users

array([109772,   6952,  65742, ...,  66406,  13788,  80826],
      shape=(30000,))

In [9]:
# оставим данные заказов для выбранных пользователей
order_data = order_data[order_data['user_id'].isin(selected_users)]
order_data

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,1187899,1,train,11,4,8,14.0,196,1,1
1,1187899,1,train,11,4,8,14.0,25133,2,1
2,1187899,1,train,11,4,8,14.0,38928,3,1
3,1187899,1,train,11,4,8,14.0,26405,4,1
4,1187899,1,train,11,4,8,14.0,39657,5,1
...,...,...,...,...,...,...,...,...,...,...
1384612,272231,206209,train,14,6,14,30.0,40603,4,0
1384613,272231,206209,train,14,6,14,30.0,15655,5,0
1384614,272231,206209,train,14,6,14,30.0,42606,6,0
1384615,272231,206209,train,14,6,14,30.0,37966,7,0


In [10]:
# теперь оставим только топ-200 продуктов

top_products = order_data['product_id'].value_counts().head(200).index
order_data = order_data[order_data['product_id'].isin(top_products)]
order_data

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,1187899,1,train,11,4,8,14.0,196,1,1
8,1187899,1,train,11,4,8,14.0,27845,9,0
9,1187899,1,train,11,4,8,14.0,49235,10,1
82,1094988,9,train,4,6,10,30.0,26604,5,1
91,1094988,9,train,4,6,10,30.0,33754,14,1
...,...,...,...,...,...,...,...,...,...,...
1384532,2951304,206198,train,8,6,12,30.0,44142,10,0
1384534,2951304,206198,train,8,6,12,30.0,15937,12,0
1384535,2951304,206198,train,8,6,12,30.0,44683,13,0
1384553,2585586,206199,train,20,2,16,30.0,35221,18,0


#### Теперь сделаем маппинги индексов товаров и пользователей

In [11]:
unique_users = sorted(order_data['user_id'].unique())
unique_products = sorted(order_data['product_id'].unique())

user_to_idx = {user_id: idx for idx, user_id in enumerate(unique_users)}
idx_to_user = {idx: user_id for user_id, idx in user_to_idx.items()}

product_to_idx = {product_id: idx for idx, product_id in enumerate(unique_products)}
idx_to_product = {idx: product_id for product_id, idx in product_to_idx.items()}

order_data['user_idx'] = order_data['user_id'].map(user_to_idx)
order_data['product_idx'] = order_data['product_id'].map(product_to_idx)

order_data

/tmp/ipykernel_16/551936385.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  order_data['user_idx'] = order_data['user_id'].map(user_to_idx)
/tmp/ipykernel_16/551936385.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  order_data['product_idx'] = order_data['product_id'].map(product_to_idx)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,user_idx,product_idx
0,1187899,1,train,11,4,8,14.0,196,1,1,0,0
8,1187899,1,train,11,4,8,14.0,27845,9,0,0,102
9,1187899,1,train,11,4,8,14.0,49235,10,1,0,198
82,1094988,9,train,4,6,10,30.0,26604,5,1,1,95
91,1094988,9,train,4,6,10,30.0,33754,14,1,1,127
...,...,...,...,...,...,...,...,...,...,...,...,...
1384532,2951304,206198,train,8,6,12,30.0,44142,10,0,26241,174
1384534,2951304,206198,train,8,6,12,30.0,15937,12,0,26241,48
1384535,2951304,206198,train,8,6,12,30.0,44683,13,0,26241,177
1384553,2585586,206199,train,20,2,16,30.0,35221,18,0,26242,133


#### Построим матрицу взаимодействий

In [12]:
n_users = len(unique_users)
n_products = len(unique_products)

In [13]:
interaction_counts = order_data.groupby(['user_idx', 'product_idx']).size()
rows = interaction_counts.index.get_level_values(0)
cols = interaction_counts.index.get_level_values(1)
data = interaction_counts.values
data

array([1, 1, 1, ..., 1, 1, 1], shape=(117148,))

In [14]:
user_item_matrix = sp.csr_matrix(
    (data, (rows, cols)),
    shape = (n_users, n_products)
)

sparsity = 1 - (user_item_matrix.nnz / (n_users * n_products))
sparsity

0.9776809937509526

### Реализуем train/test split (80/20 для каждого пользователя)

In [15]:
def create_train_test_split(order_data, user_to_idx, product_to_idx, 
                            n_users, n_products, test_ratio=0.2):
    """
    Create train/test split by holding out products from each user's order.
    
    Since Instacart train data has 1 order per user, we split the products
    within that order: 80% of products to train, 20% to test.
    
    Parameters:
    -----------
    order_data : DataFrame
        Order data with user_id, product_id columns
    user_to_idx : dict
        Mapping from user_id to matrix index
    product_to_idx : dict
        Mapping from product_id to matrix index
    n_users : int
        Number of users
    n_products : int
        Number of products
    test_ratio : float
        Fraction of products to hold out for testing
    
    Returns:
    --------
    train_matrix, test_matrix : scipy sparse matrices
    """
    
    train_rows, train_cols, train_data = [], [], []
    test_rows, test_cols, test_data = [], [], []
    
    # For each user, split their products into train/test
    for user_id, user_data in order_data.groupby('user_id'):
        user_idx = user_to_idx.get(user_id)
        if user_idx is None:
            continue
        
        # Get all products this user bought
        user_products = user_data['product_id'].values
        n_user_products = len(user_products)
        
        # Need at least 2 products to split
        if n_user_products < 2:
            # If only 1 product, put it in training
            product_idx = product_to_idx.get(user_products[0])
            if product_idx is not None:
                train_rows.append(user_idx)
                train_cols.append(product_idx)
                train_data.append(1)
            continue
        
        # Randomly shuffle and split
        shuffled_products = np.random.permutation(user_products)
        n_test = max(1, int(n_user_products * test_ratio))
        
        test_products = shuffled_products[:n_test]
        train_products = shuffled_products[n_test:]
        
        # Add to train set
        for product_id in train_products:
            product_idx = product_to_idx.get(product_id)
            if product_idx is not None:
                train_rows.append(user_idx)
                train_cols.append(product_idx)
                train_data.append(1)
        
        # Add to test set
        for product_id in test_products:
            product_idx = product_to_idx.get(product_id)
            if product_idx is not None:
                test_rows.append(user_idx)
                test_cols.append(product_idx)
                test_data.append(1)
    
    # Build sparse matrices
    train_matrix = sp.csr_matrix(
        (train_data, (train_rows, train_cols)),
        shape=(n_users, n_products)
    )
    
    test_matrix = sp.csr_matrix(
        (test_data, (test_rows, test_cols)),
        shape=(n_users, n_products)
    )
    
    # Convert to binary
    train_matrix = (train_matrix > 0).astype(int)
    test_matrix = (test_matrix > 0).astype(int)
    
    print(f"  Train: {train_matrix.nnz:,} interactions")
    print(f"  Test: {test_matrix.nnz:,} interactions")
    
    return train_matrix, test_matrix

train_matrix, test_matrix = create_train_test_split(
    order_data, user_to_idx, product_to_idx,
    n_users, n_products
)

  Train: 92,644 interactions
  Test: 24,504 interactions


#### Сохраним полученные матрицы и маппинги

In [16]:
sp.save_npz(OUTPUT_DIR / 'user_item_matrix.npz', user_item_matrix)
sp.save_npz(OUTPUT_DIR / 'train_matrix.npz', train_matrix)
sp.save_npz(OUTPUT_DIR / 'test_matrix.npz', test_matrix)

with open(OUTPUT_DIR / 'user_mapping.pkl', 'wb') as f:
    pickle.dump({'user_to_idx': user_to_idx, 'idx_to_user': idx_to_user}, f)

with open(OUTPUT_DIR / 'product_mapping.pkl', 'wb') as f:
    pickle.dump({'product_to_idx': product_to_idx, 'idx_to_product': idx_to_product}, f)